In [1]:
import sqlite3

In [2]:
# loodud tabelid
vp_data_db = "../example_data/vp_data_actors.db"

STAT = 'alati' #'mitte_kunagi'

Verbimustrite tabelite põhjal luuakse uus tabel **pattern_support**, kus on toodud verbimustrite esinemissagedused transaktsioonide andmebaasis. Tabelis on järgnevad veerud:

    pat_id - mustri ID
    verb_word - (pea)verb
    verb_compound - verbikonstruktsiooni muu osa
    phrase_case - kääne
    adp - kaassõna
    inf_verb - infiniitverb
    verb_occurrence_count - verbi(konstruktsiooni) esinemissagedus transaktsioonide andmebaasis
    absolute_support - mustri esinemissagedus transaktsioonide andmebaasis
    relative_support - mustri esinemissagedus võrreldes verbi esinemissagedusega transaktsioonide andmebaasis
    
  

In [3]:
con = sqlite3.connect(vp_data_db)
cur = con.cursor()

In [4]:
%%time

cur.execute("""
DROP TABLE IF EXISTS pattern_support_actors_{stat}
""".format(stat=STAT))

cur.execute("""
CREATE TABLE pattern_support_actors_{stat} AS
SELECT
    pm.pat_id,
    verb_word,
    verb_compound,
    phrase_case,
    adp,
    inf_verb,
    verb_match_count AS verb_occurrence_count,
    phrase_count AS absolute_support,
    CAST(phrase_count AS REAL) / CAST(verb_match_count AS REAL) * 100 AS relative_support
FROM
(
    SELECT 
        pat.pat_id,
        verb_word,
        verb_compound,
        phrase_case,
        adp,
        inf_verb,
        count(*) AS verb_match_count
    FROM
        verb_matches_actors_{stat2} as vm
    INNER JOIN
        patterns_actors_{stat3} as pat
    ON
        pat.pat_id = vm.pat_id
    GROUP BY
        pat.pat_id
) as tbl
INNER JOIN
    patterns_meta_actors_{stat4} as pm
ON
    tbl.pat_id = pm.pat_id
ORDER BY
    relative_support DESC
""".format(stat=STAT, stat2=STAT, stat3=STAT, stat4=STAT))

CPU times: user 4.05 ms, sys: 1.21 ms, total: 5.26 ms
Wall time: 13.2 ms


In [5]:
con.close()